# 04 - Genomic Trends

**Question:** How have genomic evaluations changed across birth-year cohorts, and how do
pedigree and genomic inbreeding compare?

All values are published genetic evaluations representing predicted genetic merit, not measured performance. Most are genomic evaluations, but other proof sources (GPA, GEBV, SMX PA, EBV, PA) are also present. Cohort means from one
herd on a single evaluation date; not a modelled genetic trend. Cohorts before 2018
excluded (n=1 each). The public repo ships only `data/sample_synthetic.csv`, which
reproduces structure but not correlations, so these findings reproduce only on the real
export.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd, numpy as np
from scipy import stats
hits=[p for base in [Path("."),Path(".."),Path("../..")] if base.exists()
      for p in base.rglob("Master_Database*.xlsx")]
df=pd.read_excel(hits[0],sheet_name="Data",header=1).dropna(how="all") if hits else pd.read_csv("../data/sample_synthetic.csv")
df.columns=[c.strip() for c in df.columns]
df["year"]=pd.to_datetime(df["Birth Date"],errors="coerce").dt.year
for c in ["LPI","Pro$","Fat (kg)","Genomic Inb. %","Inb.%"]:
    df[c]=pd.to_numeric(df[c],errors="coerce")
print("Loaded",df.shape)

Loaded (668, 139)


## 1. Selection indexes and production across cohorts

In [2]:
sub=df[df.year>=2018]
# Report ABSOLUTE change in predicted merit (percentages omitted: LPI/Pro$ are index
# scales with no natural ratio interpretation; r2 here is descriptive fit to annual means).
for c in ["LPI","Pro$","Fat (kg)"]:
    s=sub.groupby("year")[c].mean().dropna(); r=stats.linregress(s.index,s.values)
    print(f"{c:10s} {s.iloc[0]:7.1f} -> {s.iloc[-1]:7.1f}  slope={r.slope:+7.1f}/yr  absolute change={s.iloc[-1]-s.iloc[0]:+7.1f}")

LPI         2919.1 ->  3492.3  slope=  +71.9/yr  absolute change= +573.2
Pro$         658.2 ->  1608.8  slope= +110.0/yr  absolute change= +950.7
Fat (kg)      11.8 ->    61.3  slope=   +5.7/yr  absolute change=  +49.4


**Result.** Mean genomic merit rose across cohorts: LPI 2,919 to 3,492 (+573 points), Pro$ 658
to 1,609 (+951), Fat 11.8 to 61.3 kg (r2=0.95). This describes cohort means, not a
modelled genetic trend, and does not by itself establish that a specific intentional
strategy produced it.

## 2. Inbreeding: pedigree vs genomic, same animals and same period

Earlier drafts compared a genomic slope over 2021-2026 against a pedigree slope over a
longer window, and claimed genomic rose "53% faster". That comparison used different
periods and is invalid. Here both measures are computed on **the same 489 animals** that
carry both values, over **the same 2021+ period**, and the difference in slopes is tested
directly.

In [3]:
d=df[df["Genomic Inb. %"].notna()&df["Inb.%"].notna()].copy()
d["gap"]=d["Genomic Inb. %"]-d["Inb.%"]
dp=d[d.year>=2021]
g=dp.groupby("year")[["Genomic Inb. %","Inb.%","gap"]].mean()
print(g.round(3).to_string(),"\n")
gs=stats.linregress(g.index,g["Genomic Inb. %"]); ps=stats.linregress(g.index,g["Inb.%"]); gaps=stats.linregress(g.index,g["gap"])
print(f"genomic slope  = {gs.slope:+.3f}/yr (p={gs.pvalue:.4f})")
print(f"pedigree slope = {ps.slope:+.3f}/yr (p={ps.pvalue:.4f})")
print(f"gap slope      = {gaps.slope:+.3f}/yr (p={gaps.pvalue:.4f})  <- {'significant' if gaps.pvalue<0.05 else 'NOT significant'}")

      Genomic Inb. %  Inb.%    gap
year                              
2021          10.180  6.620  3.560
2022          11.566  8.082  3.485
2023          11.918  8.725  3.192
2024          12.330  8.890  3.440
2025          12.595  8.625  3.970
2026          13.370  9.027  4.343 

genomic slope  = +0.556/yr (p=0.0022)
pedigree slope = +0.395/yr (p=0.0440)
gap slope      = +0.161/yr (p=0.1067)  <- NOT significant


**Result (same animals, same period).** Genomic inbreeding rose +0.556/yr and pedigree
+0.395/yr over 2021-2026. Genomic increased numerically faster, but the annual increase
in their **difference** was **not statistically significant** (+0.161/yr, p = 0.107). The
claim that one rises "53% faster" is not supported and has been removed.

## 3. Bland-Altman: how much do the two measures disagree per animal?

In [4]:
paired_r=stats.pearsonr(d["Inb.%"],d["Genomic Inb. %"])[0]
mean_gap=d["gap"].mean(); sd_gap=d["gap"].std()
loa=(mean_gap-1.96*sd_gap, mean_gap+1.96*sd_gap)
safe=int(((d["Inb.%"]<8)&(d["Genomic Inb. %"]>=8)).sum())
print(f"n paired = {len(d)}")
print(f"paired correlation r = {paired_r:.3f}  (squared correlation about {paired_r**2:.2f}, substantial but incomplete)")
print(f"Bland-Altman mean difference (genomic - pedigree) = {mean_gap:.2f} points")
print(f"95% limits of agreement = [{loa[0]:.2f}, {loa[1]:.2f}]")
print(f"animals below 8% on pedigree but at/above 8% on genomic = {safe}")

n paired = 489
paired correlation r = 0.684  (squared correlation about 0.47, substantial but incomplete)
Bland-Altman mean difference (genomic - pedigree) = 3.68 points
95% limits of agreement = [-1.33, 8.69]
animals below 8% on pedigree but at/above 8% on genomic = 210


**Result.** The two measures correlate r = 0.68 (the squared correlation is about 0.47, indicating substantial but incomplete shared linear variation), so they capture **related but different quantities**: pedigree estimates
*expected* inbreeding from known relationships, genomic estimates *realized* homozygosity
from markers. The mean genomic-minus-pedigree difference is +3.68 points, with wide
limits of agreement [-1.33, 8.69]. **210 animals** sit below 8% on pedigree but at or
above 8% on genomic.

**Interpretation, stated carefully.** Pedigree is not the "wrong instrument"; both are
valid estimates of different quantities. But they disagree substantially at the
individual-animal level, and at the selected 8% threshold, 210 animals receive a different classification depending on the inbreeding measure used.